In [4]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, Any, List, Optional
import uuid

class SysMLType(Enum):
    block = "Block"
    port = "Port"
    value_property = "ValueProperty"
    requirement = "Requirement"

class ConnectorType(Enum):
    composition = "Composition"
    binding_connector = "BindingConnector"
    item_flow = "ItemFlow"
    satisfy = "Satisfy"
    trace = "Trace"

@dataclass
class SysMLPort:
    name: str
    direction: str = "inout"  # "in", "out", "inout"
    type_name: Optional[str] = None
    id: str = field(default_factory=lambda: str(uuid.uuid4()))

@dataclass
class SysMLBlock:
    name: str
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    stereotype: str = "Block"
    ports: Dict[str, SysMLPort] = field(default_factory=dict)
    values: Dict[str, Any] = field(default_factory=dict)  # e.g. {"mass": 10.5, "unit": "kg"}
    metadata: Dict[str, Any] = field(default_factory=dict)

    def add_port(self, name: str, direction: str = "inout", type_name: Optional[str] = None) -> SysMLPort:
        port = SysMLPort(name=name, direction=direction, type_name=type_name)
        self.ports[name] = port
        return port

    def to_graph_node(self) -> Dict[str, Any]:
        """Flattens element properties for easy consumption by NetworkX & Pandas."""
        return {
            "id": self.id,
            "name": self.name,
            "sysml_type": SysMLType.block.value,
            "stereotype": self.stereotype,
            "ports_count": len(self.ports),
            **self.values,
            **self.metadata,
            "_obj": self  # Reference back to the rich Python object
        }

@dataclass
class SysMLConnector:
    source_id: str
    target_id: str
    connector_type: ConnectorType
    name: Optional[str] = None
    source_port: Optional[str] = None
    target_port: Optional[str] = None
    item_property: Optional[str] = None  # e.g., "ElectricalPower"
    id: str = field(default_factory=lambda: str(uuid.uuid4()))

    def to_graph_edge(self) -> Dict[str, Any]:
        """Returns attributes structured for NetworkX edges."""
        return {
            "id": self.id,
            "name": self.name or f"{self.connector_type.value}_edge",
            "type": self.connector_type.value,
            "source_port": self.source_port,
            "target_port": self.target_port,
            "item_property": self.item_property,
            "_obj": self
        }

In [2]:
import networkx as nx
import pandas as pd

class SysMLModelBuilder:
    def __init__(self):
        self.graph = nx.DiGraph()
        self.blocks: Dict[str, SysMLBlock] = {}

    def add_block(self, block: SysMLBlock) -> SysMLBlock:
        self.blocks[block.id] = block
        self.graph.add_node(block.id, **block.to_graph_node())
        return block

    def add_connector(self, connector: SysMLConnector):
        self.graph.add_edge(
            connector.source_id,
            connector.target_id,
            **connector.to_graph_edge()
        )

    def to_dataframe(self) -> pd.DataFrame:
        """Exports block attributes into a Pandas DataFrame for analysis."""
        nodes_data = [data for _, data in self.graph.nodes(data=True)]
        df = pd.DataFrame(nodes_data)
        if "_obj" in df.columns:
            df = df.drop(columns=["_obj"])
        return df.set_index("id")

    def update_values_from_dataframe(self, df: pd.DataFrame):
        """Syncs calculated pandas values back into NetworkX and SysML objects."""
        for block_id, row in df.iterrows():
            if block_id in self.blocks:
                block = self.blocks[block_id]
                for col in df.columns:
                    if col not in ["name", "sysml_type", "stereotype", "ports_count"]:
                        block.values[col] = row[col]
                        self.graph.nodes[block_id][col] = row[col]

In [6]:
# Instantiate model builder
model = SysMLModelBuilder()

# Define Blocks
engine = model.add_block(SysMLBlock(
    name="RocketEngine",
    values={"mass_kg": 250.0, "thrust_kn": 1200.0}
))
tank = model.add_block(SysMLBlock(
    name="FuelTank",
    values={"mass_kg": 150.0, "capacity_l": 5000.0}
))

# Define Ports
p1 = engine.add_port("fuel_in", direction="in")
p2 = tank.add_port("fuel_out", direction="out")

# Connect Blocks
flow = SysMLConnector(
    source_id=tank.id,
    target_id=engine.id,
    connector_type=ConnectorType.item_flow,
    source_port=p2.name,
    target_port=p1.name,
    item_property="Kerosene"
)
model.add_connector(flow)

# Analyze with Pandas
df = model.to_dataframe()
print(df[["name", "mass_kg"]])

# Query with NetworkX
print("\nUpstream components feeding engine:")
for source in model.graph.predecessors(engine.id):
    print(model.graph.nodes[source]["name"])

                                              name  mass_kg
id                                                         
c040831b-9d7d-45b2-bfda-0d2a12c4104a  RocketEngine    250.0
f9ab8ee8-dfbd-4d72-913e-67b12b81b2ba      FuelTank    150.0

Upstream components feeding engine:
FuelTank
